# 02 · Warm-start diagnostics (Chapter 7, Section 7.3)

| Output | Thesis | Source (`Methodolody_claims_verifications_for_chapter_5`) |
|---|---|---|
| control table: baseline / rarity hint / full model | Table 7.10 | cell 6 |
| META-DDIE frequency groups | Table 7.9 | cell 33, part 2 |
| rare nodes with / without a same-class neighbour | Table 7.11 | cell 14 |
| kNN label purity of the retrieved neighbours | Ch5 claim (24.5% vs 10.7%) | cell 5 |

Needs the baseline and concat checkpoints from notebook 01 (or the thesis checkpoints on the full graph).
Left out on purpose: the concat-vs-set fusion ablation, whose saved predictions came from a split overlapping training.

**Open issue:** the group sizes printed in Table 7.9 of the thesis (216,415 / 176) do not match what this code produces on the thesis checkpoints (166,570 / 39 in the original run). Check the output below before reusing that table.

In [ ]:
DATA_ROOT = "<DATA_ROOT: e.g. /home/you/ddi_data>"   # PLACEHOLDER: thesis inputs (read only)
WORK_ROOT = "<WORK_ROOT: e.g. /home/you/ddi_runs>"   # PLACEHOLDER: new outputs
TEST_MODE = True     # True: small test subgraph from notebook 00, few epochs
                     # False: full thesis graph (needs about 64 GB of RAM)
N_TEST_DRUGS = 800   # must match the test graph built in notebook 00

In [ ]:
import os, sys
# The package is installed in the venv by setup/setup_local.sh (editable install).
# Run this notebook with the "ddi (5090)" kernel.
try:
    import ddi
except ImportError:
    raise SystemExit('ddi is not importable: run "bash setup/setup_local.sh" and select '
                     'the "ddi (5090)" kernel.')
REPO_DIR = os.path.abspath(os.path.join(os.path.dirname(ddi.__file__), "..", ".."))
print("python:", sys.executable)
print("ddi from:", os.path.dirname(ddi.__file__))

In [ ]:
import json, gc
import numpy as np
import torch
import dgl
import warnings
warnings.filterwarnings("ignore", message="Dataloader CPU affinity opt is not enabled")

from ddi.paths import get_paths
paths = get_paths(DATA_ROOT, WORK_ROOT)
paths.make_output_dirs()
paths.show()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| TEST_MODE:", TEST_MODE)

In [ ]:
from ddi import graph as G
from ddi.splits import filter_and_split_graph, save_split, check_split

if TEST_MODE:
    pair = os.path.join(paths.test_graph_dir, f"test_pair_{N_TEST_DRUGS}.bin")
    assert os.path.exists(pair), "Run notebook 00 first to build the test subgraph."
    subgraph, line_subgraph = G.load_graph_pair(pair)
else:
    G.ensure_graph_bin(paths.graph_zip, paths.graph_bin)
    subgraph, line_subgraph = G.load_graph_pair(paths.graph_bin)   # 640M edges: high-RAM only
print(subgraph); print(line_subgraph)

line_subgraph = G.attach_endpoint_features(subgraph, line_subgraph)
g_lg_filt = filter_and_split_graph(line_subgraph, label_key="label")
del line_subgraph; gc.collect()

# write the split once, then check every later run against it
RUN_TAG = f"test{N_TEST_DRUGS}" if TEST_MODE else "full"
split_path = os.path.join(paths.warm_dir, f"split_{RUN_TAG}.pt")
if os.path.exists(split_path):
    check_split(g_lg_filt, split_path)
else:
    save_split(g_lg_filt, split_path)
NUM_CLASSES = int(g_lg_filt.ndata["label"].max().item() + 1)
print("NUM_CLASSES:", NUM_CLASSES)

In [ ]:
from ddi.experiments import diagnostics as D

USE_THESIS_CHECKPOINTS = False
if USE_THESIS_CHECKPOINTS:
    assert not TEST_MODE
    ck = paths.thesis_warm_ckpt_dir
    knn_cache = paths.thesis_knn_cache_k5
else:
    ck = os.path.join(paths.warm_dir, f"checkpoints_{RUN_TAG}")
    knn_cache = os.path.join(paths.warm_dir, f"knn_{RUN_TAG}", "knn_cache_k5.pt")

npz = os.path.join(paths.diag_dir, f"control_preds_{RUN_TAG}{'_thesis' if USE_THESIS_CHECKPOINTS else ''}.npz")
if os.path.exists(npz):
    ctrl = dict(np.load(npz))
else:
    ctrl = D.control_logits(g_lg_filt, os.path.join(ck, "ckpt_baseline.pt"),
                            os.path.join(ck, "ckpt_proto_concat_rare_only_k5.pt"),
                            device, knn_cache_path=knn_cache)
    np.savez(npz, **ctrl)
train_counts, tail = D.train_counts_and_tail(g_lg_filt, NUM_CLASSES, thr=15)
print(f"rare classes: {int(tail.sum())} | rare test interactions: {int(tail[ctrl['y_true']].sum())}")

## Table 7.10: rarity-signal control

In [ ]:
t710 = D.control_table(ctrl, tail, NUM_CLASSES)

## Table 7.9: META-DDIE frequency groups

In [ ]:
t79 = D.frequency_groups(ctrl, NUM_CLASSES)

## Table 7.11: same-class neighbour split

In [ ]:
cache = torch.load(knn_cache, map_location="cpu")
t711 = D.label_dependence(ctrl, g_lg_filt, cache["knn_idx"], cache["knn_sim"], train_counts)

## kNN label purity

In [ ]:
purity = D.knn_purity(g_lg_filt, cache["knn_idx"], cache["knn_sim"])
json.dump({"table_7_10": t710, "table_7_9": t79, "table_7_11": {str(k): v for k, v in t711.items()},
           "knn_purity": purity},
          open(os.path.join(paths.diag_dir, f"diagnostics_{RUN_TAG}.json"), "w"), indent=2)